# AMDT rebuttal pipeline — Colab reproduction

One-click reproduction of the figures and tables that answer Reviewer 1's
decision letter for **JST-6070-2025.R1**.

**Runtime:** set *Runtime → Change runtime type → GPU* before running the CNN
section. Everything else runs on CPU.

Sections
1. Setup and determinism
2. Data
3. Round-trip correctness (the claim everything else rests on)
4. Quality vs. all baselines — comments 4, 7
5. Decomposition ablation + payload-statistics attacks — comments 1, 2
6. SRM-subset + FLD ensemble steganalysis — comment 3
7. CNN steganalysis (GPU, Drive checkpointing, auto-resume) — comment 3
8. Significance testing — comment 4
9. Runtime and complexity — comment 6
10. Reproducibility manifest — comment 5


## 1. Setup and determinism

`set_seed` must run before any CUDA context exists, so it is the first thing after the imports.

In [ ]:
#@title Install (Colab only)
import importlib, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "hydra-core", "omegaconf", "PyWavelets"], check=True)
print("colab:", IN_COLAB)

In [ ]:
#@title Locate the project and lock all randomness
from pathlib import Path
import os, sys

# Point PROJECT at the amdt_python directory (clone it or mount Drive).
PROJECT = Path("/content/amdt_python") if IN_COLAB else Path.cwd().parent
sys.path.insert(0, str(PROJECT / "src"))

from amdt.utils.seeding import set_seed, seeded_rng, SeedPolicy
set_seed(0)
print(SeedPolicy.text)

In [ ]:
#@title Optional: mount Drive for checkpoints that survive a disconnect
CKPT_DIR = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/amdt_ckpt"
    os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

## 2. Data

The 29 benchmark covers by default. For the steganalysis table that goes in the paper, stage BOSSBase 1.01 to local disk first (reading 10k images straight off Drive dominates epoch time).

In [ ]:
from amdt.data.dataset import load_dataset, payload_bits_for_rate, describe_payloads

IMAGES_DIR = PROJECT.parent / "baseline code matlab" / "Images"
images, spec = load_dataset(IMAGES_DIR, side=512)
print(f"{spec.n_images} covers, {spec.side}x{spec.side}")
if spec.steganalysis_warning():
    print("\nCAVEAT:", spec.steganalysis_warning())

In [ ]:
#@title BOSSBase staging (skip if using the local set)
# import shutil
# LOCAL = Path("/content/bossbase")
# if not LOCAL.exists():
#     shutil.copy("/content/drive/MyDrive/BOSSbase_1.01.zip", "/content/boss.zip")
#     shutil.unpack_archive("/content/boss.zip", LOCAL)
# images, spec = load_dataset(LOCAL, side=512, pattern="*.pgm")

## 3. Round-trip correctness

If extraction is not exact, every quality number is meaningless. This is checked here and asserted inside the pipeline for every embedding.

In [ ]:
import numpy as np
from amdt.ga.optimizer import GAConfig, search_space_size
from amdt.stego.amdt import run_amdt, capacity_bits
from amdt.stego.codec import extract

KEY = bytes.fromhex("2a7f4c9e1b6d80f3a5c2e7091d4b8fa6"
                    "3e5c1908d7b2a4f60c3e819d5a7b2f4c")
cover = images[0]
L = payload_bits_for_rate(cover.shape, 0.1)
payload = seeded_rng(0, "demo").integers(0, 2, L, dtype=np.uint8)

res = run_amdt(cover, payload, KEY,
               GAConfig(population=25, generations=60, patience=20, n_segments=4),
               seeded_rng(0, "ga"))
recovered, genes = extract(res.stego, KEY)

print(f"search space / segment : {search_space_size(1):.3e}")
print(f"payload                : {L} bits ({L/cover.size:.3f} bpp)")
print(f"max capacity           : {capacity_bits(cover.shape, 4, 15)} bits")
print(f"PSNR {res.quality.psnr:.2f} dB | SSIM {res.quality.ssim:.5f} | "
      f"MSE {res.quality.mse:.5f} | {res.quality.embedding_efficiency:.2f} bits/change")
print(f"exact recovery         : {np.array_equal(recovered, payload)}")
for i, g in enumerate(genes):
    print(f"  segment {i}: {g.as_dict()}")

## 4–10. Full pipeline

The studies are driven by Hydra so the notebook and the CLI run identical code. Reduce `dataset.limit` for a fast smoke run.

In [ ]:
import subprocess, sys
cmd = [sys.executable, str(PROJECT / "run_experiments.py"),
       f"dataset.root={IMAGES_DIR}",
       "experiment.seeds=[0,1,2,3,4]",
       "experiment.payload_rates_bpp=[0.05,0.1,0.2,0.4]",
       "ga.population=25", "ga.generations=100", "ga.n_segments=4",
       "experiment.studies=[quality,ablation,targeted,steganalysis,runtime,stats]"]
print(" ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT), check=True)

In [ ]:
#@title Locate the newest run and list what it produced
import json
runs = sorted((PROJECT / "outputs" / "amdt_rebuttal").glob("*"))
RUN = runs[-1]
print(RUN)
print(json.dumps(json.loads((RUN / "index.json").read_text()), indent=2)[:1500])

In [ ]:
#@title Headline tables
import pandas as pd
q = pd.read_csv(RUN / "results" / "quality.csv")
ref = sorted(q.rate_bpp.unique())[len(q.rate_bpp.unique()) // 2]
print(f"--- imperceptibility @ {ref} bpp ---")
print(q[q.rate_bpp == ref].groupby("method")[
    ["psnr", "ssim", "mse", "embedding_efficiency"]].agg(["mean", "std"]).round(4))

s = pd.read_csv(RUN / "results" / "significance.csv")
print("\n--- PSNR significance vs. AMDT (Holm-corrected) ---")
print(s[(s.rate_bpp == ref) & (s.metric == "psnr")][
    ["method_b", "mean_diff", "diff_ci_low", "diff_ci_high",
     "preferred_test", "p_adjusted", "cohens_dz", "significant"]].round(4)
    .to_string(index=False))

In [ ]:
#@title Detection results (lower accuracy / higher P_E favours the stego method)
d = pd.read_csv(RUN / "results" / "steganalysis_classical.csv")
print(d[d.rate_bpp == ref][
    ["method", "detector", "accuracy", "precision", "recall", "f1",
     "auc", "p_e", "md_at_fa5"]].round(3).to_string(index=False))

In [ ]:
#@title Show the generated figures
from IPython.display import display
import matplotlib.image as mpimg, matplotlib.pyplot as plt
for name in ["fig02_psnr", "fig05_laplacian", "fig10_roc",
             "fig11_detectability", "fig13_convergence"]:
    p = RUN / "figures" / f"{name}.svg"
    if p.exists():
        from IPython.display import SVG
        print(name); display(SVG(filename=str(p)))

## 7. CNN steganalysis (GPU)

Requires PyTorch and a CUDA device. Checkpoints go to Drive and the trainer auto-resumes, so a disconnected session loses nothing. SRNet converges far better with a curriculum: train at 0.4 bpp first, then pass that checkpoint via `steganalysis.curriculum_from`.

In [ ]:
import torch
assert torch.cuda.is_available(), "switch the runtime to GPU"
print(torch.cuda.get_device_name(0))

cmd = [sys.executable, str(PROJECT / "run_experiments.py"),
       f"dataset.root={IMAGES_DIR}",
       "steganalysis=cnn", "steganalysis.model=yedroudj",
       "steganalysis.epochs=100",
       f"steganalysis.checkpoint_dir={CKPT_DIR}",
       "experiment.payload_rates_bpp=[0.4]",
       "experiment.studies=[quality,cnn]"]
subprocess.run(cmd, cwd=str(PROJECT), check=True)

## 11. Tracking, provenance and the training supervisor

Optional, and off by default. Enable them when you want the run recorded and
the GitHub history to be the record of how a number was reached.

In [ ]:
#@title GitHub PAT from Colab Secrets — never pasted into a cell
# Colab: key icon in the left sidebar -> add secret named GITHUB_PAT.
# Scope it to this one repository, Contents: read/write, expiry <= 90 days.
from amdt.utils.repo import GitRepo, load_pat

GIT_REMOTE = None  #@param {type:"string"}  e.g. https://github.com/<owner>/<repo>.git
token = load_pat()
print("PAT found:", bool(token))   # never print the token itself

repo = None
if GIT_REMOTE:
    repo = GitRepo(PROJECT, remote=GIT_REMOTE)
    repo.init()
    repo.assert_no_token_in_config()   # fails loudly if a token ever leaks in
    print("HEAD:", repo.head())

In [ ]:
#@title W&B + git provenance run
cmd = [sys.executable, str(PROJECT / "run_experiments.py"),
       f"dataset.root={IMAGES_DIR}",
       "tracking=wandb", "tracking.project=amdt-steganography",
       f"tracking.git.enabled={bool(GIT_REMOTE)}",
       f"tracking.git.push={bool(GIT_REMOTE and token)}",
       "experiment.studies=[quality,stats]"]
subprocess.run(cmd, cwd=str(PROJECT), check=True)
# Tables land in paper/tables/ at a stable path, so the manuscript can
# \input{} them and a re-run updates the paper in place.

In [ ]:
#@title Optuna — tune the *attacker*, on validation only
cmd = [sys.executable, str(PROJECT / "run_experiments.py"),
       f"dataset.root={IMAGES_DIR}",
       "search.enabled=true", "search.objective=cnn", "search.direction=minimize",
       "search.n_trials=25",
       f"search.storage=sqlite:///{CKPT_DIR}/optuna.db" if CKPT_DIR
       else "search.storage=sqlite:///optuna.db",
       "experiment.studies=[]"]
subprocess.run(cmd, cwd=str(PROJECT), check=True)
# A weak detector is not evidence of security, so the search minimises the
# detector's P_E — it makes the attacker as strong as it can.

### Supervised CNN run

Start the watcher **on your own machine**, not here — it has to outlive this
runtime:

```bash
python watch_training.py --run <entity>/<project>/<run_id> \
    --remote https://github.com/<owner>/<repo>.git --baseline 0.45
```

Then run the cell below. The stub polls `agent/patches` between epochs and
applies whatever the watcher published. Budget: three rounds, counted in the
checkpoint so a disconnect cannot reset it.

In [ ]:
#@title CNN training with the patch stub enabled
cmd = [sys.executable, str(PROJECT / "run_experiments.py"),
       f"dataset.root={IMAGES_DIR}",
       "steganalysis=cnn", "steganalysis.model=yedroudj",
       f"steganalysis.checkpoint_dir={CKPT_DIR}",
       "tracking=wandb", "supervisor.enabled=true",
       "experiment.payload_rates_bpp=[0.4]",
       "experiment.studies=[quality,cnn]"]
subprocess.run(cmd, cwd=str(PROJECT), check=True)

In [ ]:
#@title Any method-altering edits? Read this before writing the methods section.
from amdt.experiments.supervisor import Supervisor
sup = Supervisor(RUN)
altering = sup.method_altering_summary()
print(f"{len(sup.interventions())} intervention(s), {len(altering)} method-altering")
for i in altering:
    print(f"  round {i['round']} [{i['trigger']}] {i['diff_summary']}")
if altering:
    print("\nThe manuscript describes what you designed. After these edits it "
          "may not describe what ran \u2014 reconcile before submission.")

## 12. Execution target and Kaggle data

Two things that decide whether a number is quotable: which machine produced it,
and which bytes it was computed on.

In [ ]:
#@title Declare the target explicitly — never let it be inferred
from amdt.utils.execution import configure_execution

# In Colab this notebook is a hosted runtime, so its timings are not reportable.
# Metrics are fine; wall-clock is not, because the VM changes between sessions.
target = configure_execution(name="colab_gpu" if IN_COLAB else "local",
                             device="cuda" if IN_COLAB else "cpu",
                             reported=True, threads=1, n_jobs=1)
print(target.label)
print("timings reportable:", target.timing_valid, target.timing_invalid_reason or "")

In [ ]:
#@title Kaggle credentials from Secrets — never typed into a cell
# Colab: key icon -> add KAGGLE_USERNAME and KAGGLE_KEY.
# Locally: ~/.kaggle/kaggle.json, chmod 600.
from amdt.data.kaggle import load_credentials, download_dataset

try:
    creds = load_credentials()
    print("credentials:", creds.describe())   # the key itself never prints
except RuntimeError as e:
    print(e)

In [ ]:
#@title Fetch BOSSBase, pinned and hash-verified
SLUG = "lolrudy/bossbase"  #@param {type:"string"}
path, pin = download_dataset(SLUG)
print(path, pin.n_files, "files")
print("version:", pin.version, "sha256:", (pin.sha256 or "")[:16])
print("\nPaste these into configs/dataset/bossbase_kaggle.yaml under `pin:`.")
print("Every later run compares against them and warns on drift \u2014 it will")
print("never silently upgrade you to a dataset that gained rows.")

In [ ]:
#@title Run on BOSSBase (subsample — 10k covers x 10 methods is ~100 h)
cmd = [sys.executable, str(PROJECT / "run_experiments.py"),
       "dataset=bossbase_kaggle", "dataset.limit=2000",
       "env=colab_gpu" if IN_COLAB else "env=local",
       "'experiment.studies=[quality,steganalysis,stats]'"]
print(" ".join(cmd))
# subprocess.run(cmd, cwd=str(PROJECT), check=True)

## 13. Run-folder sync to Drive

Mount once, here at the top. One OAuth approval per session is Google's consent
step; after it every run mirrors itself automatically, with no further prompts.

In [ ]:
#@title Mount Drive once — the mounted path becomes the sync target
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")     # single OAuth approval per session
    print("mounted")

from amdt.utils.drive_sync import resolve_drive_root
print("sync target:", resolve_drive_root())

In [ ]:
#@title Any run with sync=drive now mirrors itself while it runs
cmd = [sys.executable, str(PROJECT / "run_experiments.py"),
       f"dataset.root={IMAGES_DIR}", "sync=drive",
       "'experiment.studies=[quality,stats]'"]
print(" ".join(cmd))
# subprocess.run(cmd, cwd=str(PROJECT), check=True)
# Artifacts are staged locally and moved across atomically, so a disconnect
# mid-run leaves complete files in Drive, never half-uploaded ones.

## Reading the results honestly

* `P_E = 0.5` means undetectable; **lower detector accuracy is better** for the
  steganographic method. The plots are labelled to prevent the direction being
  misread.
* `results/steganalysis_caveat.json` records the corpus-size warning whenever
  the dataset is too small for a credible detection claim.
* `results/significance_flagged_small_effects.csv`, if present, lists
  comparisons that reached significance while the effect was smaller than the
  seed-to-seed SD. Do not report those as improvements.
* T3/T4 defeat payload-statistics attacks (χ², pairs-of-values), **not**
  residual-based detectors. See `tables/tab_targeted.tex`.